# BÀI THỰC HÀNH: THƯ VIỆN PANDAS TRONG PYTHON

**Môn học:** Khoa học Dữ liệu / Lập trình Python  
**Giảng viên:** ThS. Lê Nhật Tùng  
**Thời gian:** 180 phút  
**File dữ liệu:** `du_lieu_thuc_hanh.xlsx` (3 sheet: `nhan_vien`, `don_hang`, `du_lieu_loi`)

---

## Giới thiệu

Pandas là thư viện xử lý dữ liệu hàng đầu trong Python, được sử dụng rộng rãi trong phân tích dữ liệu, khoa học dữ liệu và học máy. Pandas cung cấp hai cấu trúc dữ liệu chính:

- **Series**: mảng một chiều có nhãn chỉ mục (index)
- **DataFrame**: bảng dữ liệu hai chiều, giống bảng tính Excel hoặc bảng trong cơ sở dữ liệu quan hệ

---

## Mục tiêu bài thực hành

Sau khi hoàn thành, sinh viên có khả năng:

1. Đọc, ghi dữ liệu từ nhiều định dạng file khác nhau
2. Kiểm tra, hiểu cấu trúc và kiểu dữ liệu của DataFrame
3. Chọn lọc, lọc và sắp xếp dữ liệu theo nhiều điều kiện
4. Làm sạch dữ liệu bị lỗi, thiếu, trùng lặp
5. Tạo thêm cột tính toán, biến đổi dữ liệu
6. Thống kê mô tả và phân tích theo nhóm
7. Kết nối nhiều bảng dữ liệu (merge/join)
8. Tạo pivot table và phân tích chéo
9. Xuất kết quả ra file Excel với nhiều sheet

---

## Mô tả bộ dữ liệu

| Sheet | Mô tả | Số dòng |
|-------|-------|---------|
| `nhan_vien` | Thông tin 100 nhân viên: lương, phòng ban, đánh giá | 100 |
| `don_hang` | 200 đơn đặt hàng sản phẩm điện tử | 200 |
| `du_lieu_loi` | Dữ liệu có sự cố: null, trùng lặp, định dạng sai | 15 |

---
## PHẦN 0: Cài đặt và Kiểm tra Môi trường

In [4]:
import sys
print(f'Python: {sys.version}')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'

print(f'pandas     : {pd.__version__}')
print(f'numpy      : {np.__version__}')
print(f'matplotlib : {matplotlib.__version__}')

# Cấu hình hiển thị DataFrame
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)
pd.set_option('display.float_format', '{:,.0f}'.format)

import warnings
warnings.filterwarnings('ignore')
print('Môi trường sẵn sàng!')

Python: 3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:09:58) [MSC v.1929 64 bit (AMD64)]
pandas     : 2.3.3
numpy      : 2.3.5
matplotlib : 3.10.6
Môi trường sẵn sàng!


---
## PHẦN 1: Đọc Dữ liệu từ File Excel

### 1.1 Lý thuyết

Hàm `pd.read_excel()` là hàm chủ lực để đọc file Excel. Một số tham số quan trọng:

| Tham số | Ý nghĩa | Ví dụ |
|---------|---------|-------|
| `sheet_name` | Tên hoặc số thứ tự sheet | `'nhan_vien'` hoặc `0` |
| `header` | Hàng nào là tiêu đề cột | `0` (mặc định) |
| `dtype` | Ép kiểu dữ liệu cột | `{'Mã NV': str}` |
| `na_values` | Các giá trị được coi là NaN | `['N/A', '-', '']` |
| `usecols` | Chỉ đọc một số cột nhất định | `['A', 'C', 'E']` |
| `nrows` | Chỉ đọc một số hàng | `50` |

### 1.2 Đọc dữ liệu

In [7]:
# Đọc tất cả sheet cùng lúc — trả về dict {tên_sheet: DataFrame}
all_sheets = pd.read_excel('BaiTap02_du_lieu_thuc_hanh.xlsx', sheet_name=None)

print('Danh sách sheet trong file:')
for name, df in all_sheets.items():
    print(f'  - {name}: {df.shape[0]} dòng x {df.shape[1]} cột')

Danh sách sheet trong file:
  - nhan_vien: 100 dòng x 16 cột
  - don_hang: 200 dòng x 13 cột
  - du_lieu_loi: 15 dòng x 8 cột


In [8]:
# Đọc từng sheet riêng lẻ
df_nv = pd.read_excel(
    'BaiTap02_du_lieu_thuc_hanh.xlsx',
    sheet_name='nhan_vien',
    dtype={'Mã NV': str}         # Ép Mã NV là string, tránh bị đọc thành số
)

df_dh = pd.read_excel(
    'BaiTap02_du_lieu_thuc_hanh.xlsx',
    sheet_name='don_hang',
    dtype={'Mã NV': str, 'Mã SP': str}
)

df_loi = pd.read_excel(
    'BaiTap02_du_lieu_thuc_hanh.xlsx',
    sheet_name='du_lieu_loi',
    na_values=['N/A', '', ' ']   # Các giá trị được nhận diện là null
)

print(f'nhan_vien   : {df_nv.shape}')
print(f'don_hang    : {df_dh.shape}')
print(f'du_lieu_loi : {df_loi.shape}')

nhan_vien   : (100, 16)
don_hang    : (200, 13)
du_lieu_loi : (15, 8)


In [9]:
# Xem 5 dòng đầu của bảng nhân viên
df_nv.head()

,Mã NV,Họ đệm,Tên,Giới tính,Năm sinh,Phòng ban,Chức vụ,Lương cơ bản,Phụ cấp,Thuế TNCN,Lương thực lĩnh,Ngày vào làm,Trình độ,Tỉnh thành,Số ngày phép,Đánh giá
0,NV001,Bùi,Minh,Nam,1979,Kỹ thuật,Trưởng nhóm,6500000,1800000,345659,7954341,24/02/2020,Đại học,Cần Thơ,3,Yếu
1,NV002,Vũ,Long,Nam,1982,IT,QA,4000000,1000000,479256,4520744,15/10/2014,Cao đẳng,Hải Phòng,2,Tốt
2,NV003,Lê,Hoa,Nữ,1986,Kế toán,Kế toán trưởng,12000000,600000,1475678,11124322,28/06/2019,Tiến sĩ,Hồ Chí Minh,6,Xuất sắc
3,NV004,Ngô,Nhi,Nữ,1997,Marketing,Trưởng phòng,11000000,1400000,1633744,10766256,03/01/2020,Thạc sĩ,Hồ Chí Minh,6,Khá
4,NV005,Phan,Ngọc,Nữ,1981,Nhân sự,Chuyên viên,6000000,1000000,620484,6379516,22/05/2021,Thạc sĩ,Đà Nẵng,7,Trung bình


In [10]:
# Xem 5 dòng cuối
df_nv.tail()

,Mã NV,Họ đệm,Tên,Giới tính,Năm sinh,Phòng ban,Chức vụ,Lương cơ bản,Phụ cấp,Thuế TNCN,Lương thực lĩnh,Ngày vào làm,Trình độ,Tỉnh thành,Số ngày phép,Đánh giá
95,NV096,Nguyễn,Cường,Nam,1985,Kế toán,Kế toán trưởng,16500000,1800000,1302700,16997300,19/07/2019,THPT,Hà Nội,5,Tốt
96,NV097,Lê,Ngọc,Nữ,1982,Kế toán,Kế toán trưởng,6500000,1700000,890841,7309159,14/10/2016,Cao đẳng,Cần Thơ,5,Tốt
97,NV098,Phạm,Long,Nam,1981,Marketing,Thiết kế,8500000,1200000,512384,9187616,25/06/2015,Cao đẳng,Cần Thơ,2,Yếu
98,NV099,Phan,Hùng,Nam,1990,Hành chính,Bảo vệ,18000000,1500000,2455644,17044356,15/10/2022,Cao đẳng,Đà Nẵng,7,Xuất sắc
99,NV100,Đỗ,Thảo,Nữ,1993,Hành chính,Văn thư,6000000,1400000,577045,6822955,02/06/2018,Đại học,Hà Nội,5,Khá


In [11]:
# Lấy mẫu ngẫu nhiên 5 dòng — hữu ích khi dữ liệu lớn
df_nv.sample(5, random_state=42)

,Mã NV,Họ đệm,Tên,Giới tính,Năm sinh,Phòng ban,Chức vụ,Lương cơ bản,Phụ cấp,Thuế TNCN,Lương thực lĩnh,Ngày vào làm,Trình độ,Tỉnh thành,Số ngày phép,Đánh giá
83,NV084,Đinh,Hùng,Nam,1991,Kế toán,Kế toán viên,20000000,1200000,2890831,18309169,19/02/2017,Đại học,Hải Phòng,7,Xuất sắc
53,NV054,Đặng,Hoa,Nữ,1999,Kinh doanh,Giám sát,11000000,1700000,1045566,11654434,14/10/2019,THPT,Hải Phòng,9,Trung bình
70,NV071,Lê,Tuấn,Nam,1994,Marketing,Chuyên viên,8500000,1200000,816977,8883023,22/04/2017,Cao đẳng,Vinh,4,Xuất sắc
45,NV046,Lý,Thu,Nữ,1991,Kỹ thuật,Thực tập sinh,19000000,1900000,1777592,19122408,21/05/2019,Tiến sĩ,Hải Phòng,5,Tốt
44,NV045,Cao,Đức,Nam,1986,Kế toán,Thủ quỹ,14500000,500000,1956885,13043115,03/09/2020,THPT,Hồ Chí Minh,6,Khá


---
## PHẦN 2: Kiểm tra Cấu trúc và Kiểu Dữ liệu

### 2.1 Các phương thức kiểm tra cơ bản

Đây là bước đầu tiên **luôn luôn phải thực hiện** khi nhận một bộ dữ liệu mới.  
Mục tiêu: hiểu rõ số dòng, số cột, kiểu dữ liệu, số lượng giá trị null.

In [12]:
# Thông tin tổng quát — nên chạy đầu tiên khi khám phá dữ liệu mới
print('=== THÔNG TIN TỔNG QUÁT ===')
df_nv.info()

=== THÔNG TIN TỔNG QUÁT ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Mã NV            100 non-null    object
 1   Họ đệm           100 non-null    object
 2   Tên              100 non-null    object
 3   Giới tính        100 non-null    object
 4   Năm sinh         100 non-null    int64 
 5   Phòng ban        100 non-null    object
 6   Chức vụ          100 non-null    object
 7   Lương cơ bản     100 non-null    int64 
 8   Phụ cấp          100 non-null    int64 
 9   Thuế TNCN        100 non-null    int64 
 10  Lương thực lĩnh  100 non-null    int64 
 11  Ngày vào làm     100 non-null    object
 12  Trình độ         100 non-null    object
 13  Tỉnh thành       100 non-null    object
 14  Số ngày phép     100 non-null    int64 
 15  Đánh giá         100 non-null    object
dtypes: int64(6), object(10)
memory usage: 12.6+ KB


In [ ]:
print(f'Số dòng  : {df_nv.shape[0]}')
print(f'Số cột   : {df_nv.shape[1]}')
print(f'Tổng ô   : {df_nv.size}')

print('\nDanh sách cột và kiểu dữ liệu:')
for col in df_nv.columns:
    print(f'  {col}: {df_nv[col].dtype}')

In [ ]:
# Kiểm tra giá trị null — báo cáo đầy đủ
null_count = df_nv.isnull().sum()
null_pct   = (df_nv.isnull().sum() / len(df_nv) * 100).round(2)

null_report = pd.DataFrame({
    'Số null': null_count,
    'Phần trăm (%)': null_pct
})
print('Các cột có giá trị null:')
null_report[null_report['Số null'] > 0]

In [ ]:
# Thống kê mô tả số liệu
# include='all' hiển thị cả cột số lẫn cột chuỗi
df_nv.describe(include='all').T

In [ ]:
# Số giá trị duy nhất mỗi cột
print('Số giá trị duy nhất mỗi cột:')
print(df_nv.nunique())

In [ ]:
# Phân phối giá trị của các cột phân loại
print('=== Phân bổ theo Phòng ban ===')
print(df_nv['Phòng ban'].value_counts())

print('\n=== Phân bổ theo Đánh giá ===')
print(df_nv['Đánh giá'].value_counts())

print('\n=== Phân bổ theo Giới tính ===')
print(df_nv['Giới tính'].value_counts())

---
## PHẦN 3: Chọn lọc Dữ liệu — Indexing và Slicing

### 3.1 Chọn cột

Có hai cách chọn cột:
- `df['tên_cột']` trả về **Series**
- `df[['cột1', 'cột2']]` trả về **DataFrame**

In [ ]:
# Chọn một cột — kết quả là Series
luong = df_nv['Lương cơ bản']
print(type(luong))
print(luong.head())

In [ ]:
# Chọn nhiều cột — kết quả là DataFrame
cols = ['Mã NV', 'Họ đệm', 'Tên', 'Phòng ban', 'Lương cơ bản', 'Đánh giá']
df_nv[cols].head(8)

### 3.2 Chọn dòng bằng `.loc` và `.iloc`

| Phương thức | Ý nghĩa | Ghi chú |
|-------------|---------|----------|
| `.loc[nhãn]` | Theo nhãn chỉ mục | Bao gồm cả điểm cuối |
| `.iloc[vị_trí]` | Theo vị trí số nguyên | Không bao gồm điểm cuối |

In [ ]:
# .loc: chọn theo nhãn chỉ mục — loc[0:4] lấy dòng 0 đến 4 (5 dòng, gồm cả 4)
print('=== .loc[0:4] ===')
print(df_nv.loc[0:4, ['Mã NV', 'Họ đệm', 'Tên', 'Phòng ban']])

In [ ]:
# .iloc: chọn theo vị trí số — iloc[0:5] lấy dòng 0,1,2,3,4 (không tính dòng 5)
print('=== .iloc[0:5, 0:4] ===')
print(df_nv.iloc[0:5, 0:4])

In [ ]:
# Chọn dòng không liên tiếp
print('=== Dòng 0, 10, 20, 30 ===')
print(df_nv.iloc[[0, 10, 20, 30], :])

### 3.3 Lọc dữ liệu theo điều kiện — Boolean Indexing

Đây là kỹ thuật **quan trọng nhất**. Nguyên tắc:
1. Tạo biến điều kiện (Series True/False)
2. Dùng biến điều kiện để lọc DataFrame

Khi kết hợp nhiều điều kiện: AND dùng `&`, OR dùng `|`, NOT dùng `~`.  
**Mỗi điều kiện phải được bao bởi dấu ngoặc `()`.**

In [ ]:
# Điều kiện đơn giản — nhân viên phòng IT
dk = df_nv['Phòng ban'] == 'IT'
print(f'Nhân viên IT: {dk.sum()} người')
df_nv[dk][['Mã NV', 'Họ đệm', 'Tên', 'Chức vụ', 'Lương cơ bản']].head()

In [ ]:
# Nhiều điều kiện: nhân viên nữ, phòng IT, lương trên 15 triệu
dk_phuc_tap = (
    (df_nv['Giới tính'] == 'Nữ') &
    (df_nv['Phòng ban'] == 'IT') &
    (df_nv['Lương cơ bản'] > 15_000_000)
)
print(f'Nữ - IT - Lương > 15 triệu: {dk_phuc_tap.sum()} người')
df_nv[dk_phuc_tap][['Mã NV', 'Họ đệm', 'Tên', 'Chức vụ', 'Lương cơ bản']]

In [ ]:
# .isin() — lọc nhiều giá trị cùng lúc
phong_chon = ['IT', 'Kế toán', 'Nhân sự']
df_chon = df_nv[df_nv['Phòng ban'].isin(phong_chon)]
print(f'Nhân viên IT/Kế toán/Nhân sự: {len(df_chon)} người')
print(df_chon['Phòng ban'].value_counts())

In [ ]:
# .between() — lọc khoảng giá trị
df_sinh_80_90 = df_nv[df_nv['Năm sinh'].between(1980, 1990)]
print(f'Nhân viên sinh 1980–1990: {len(df_sinh_80_90)}')

# .str.contains() — tìm kiếm chuỗi
df_truong = df_nv[df_nv['Chức vụ'].str.contains('Trưởng', na=False)]
print(f'\nCác vị trí có "Trưởng": {len(df_truong)}')
print(df_truong['Chức vụ'].value_counts())

In [ ]:
# .query() — cách lọc ngắn gọn, giống SQL WHERE
# Dùng backtick (`) cho tên cột có khoảng trắng
df_q = df_nv.query('`Phòng ban` == "Kinh doanh" and `Lương cơ bản` > 15_000_000')
print(f'Kinh doanh + Lương > 15 triệu: {len(df_q)} người')
df_q[['Mã NV', 'Họ đệm', 'Tên', 'Lương cơ bản', 'Đánh giá']]

### Bài tập 3.1

Sử dụng DataFrame `df_nv` để thực hiện các yêu cầu sau:

1. Lọc ra tất cả nhân viên có đánh giá **'Xuất sắc'** hoặc **'Tốt'**
2. Tìm nhân viên **không thuộc** phòng IT và phòng Marketing
3. Lọc nhân viên có **lương thực lĩnh trên 20 triệu** và **có bằng Thạc sĩ**
4. Tìm nhân viên có **số ngày phép bằng 0** (chưa nghỉ phép lần nào)
5. Lọc nhân viên **ở Hà Nội** HOẶC **ở Hồ Chí Minh**, sinh **trước năm 1985**

In [ ]:
# --- KHUNG BÀI TẬP 3.1 ---

# 1. Đánh giá Xuất sắc hoặc Tốt
bai1 = ...  # TODO
# print(f'1. Đánh giá tốt: {len(bai1)} người')

# 2. Không thuộc IT và Marketing
bai2 = ...  # TODO
# print(f'2. Ngoài IT/Marketing: {len(bai2)} người')

# 3. Lương thực lĩnh > 20 triệu và Thạc sĩ
bai3 = ...  # TODO
# print(f'3. Lương>20M & Thạc sĩ: {len(bai3)} người')

# 4. Số ngày phép = 0
bai4 = ...  # TODO
# print(f'4. Chưa nghỉ phép: {len(bai4)} người')

# 5. Hà Nội hoặc HCM, sinh trước 1985
bai5 = ...  # TODO
# print(f'5. HN/HCM sinh <1985: {len(bai5)} người')

---
## PHẦN 4: Làm sạch Dữ liệu (Data Cleaning)

Đây là giai đoạn tốn nhiều thời gian nhất trong thực tế (thường chiếm 60–80% thời gian dự án).

### 4.1 Kiểm tra dữ liệu lỗi

In [ ]:
# Nhìn toàn bộ dữ liệu lỗi
print('=== DỮ LIỆU CÓ SỰ CỐ ===')
df_loi

In [ ]:
print('Thông tin cấu trúc:')
df_loi.info()
print('\nSố giá trị null theo cột:')
print(df_loi.isnull().sum())

### 4.2 Xử lý giá trị null

Có 3 chiến lược chính:

| Chiến lược | Phương thức | Dùng khi |
|-----------|-------------|----------|
| Xóa dòng | `dropna()` | Tỷ lệ null thấp, dòng không quan trọng |
| Điền giá trị | `fillna()` | Có giá trị hợp lý để thay thế |
| Nội suy | `interpolate()` | Dữ liệu chuỗi thời gian có xu hướng |

In [ ]:
# Tạo bản sao để làm sạch — không chỉnh sửa dữ liệu gốc
df_clean = df_loi.copy()

print('Trước khi xử lý null:')
print(df_clean.isnull().sum())

In [ ]:
# Xóa dòng có null ở cột Họ tên (trường bắt buộc)
df_clean = df_clean.dropna(subset=['Họ tên'])
print(f'Sau khi xóa dòng null Họ tên: {len(df_clean)} dòng')

# Điền null ở cột Tuổi bằng trung vị
trung_vi_tuoi = df_clean['Tuổi'].median()
df_clean['Tuổi'] = df_clean['Tuổi'].fillna(trung_vi_tuoi)
print(f'Điền Tuổi null bằng trung vị: {trung_vi_tuoi}')

# Điền null ở cột Lương bằng trung bình
trung_binh_luong = df_clean['Lương'].mean()
df_clean['Lương'] = df_clean['Lương'].fillna(trung_binh_luong)
print(f'Điền Lương null bằng trung bình: {trung_binh_luong:,.0f}')

# Điền null Ngày sinh bằng 'Chưa rõ'
df_clean['Ngày sinh'] = df_clean['Ngày sinh'].fillna('Chưa rõ')

print('\nSau khi xử lý null:')
print(df_clean.isnull().sum())

### 4.3 Xử lý giá trị ngoại lệ (Outlier)

**Phương pháp IQR** (Interquartile Range):
- Tính Q1 (phân vị 25%) và Q3 (phân vị 75%)
- IQR = Q3 − Q1
- Giới hạn dưới = Q1 − 1.5 × IQR
- Giới hạn trên = Q3 + 1.5 × IQR
- Giá trị nằm ngoài khoảng trên được coi là ngoại lệ

In [ ]:
# Phát hiện ngoại lệ ở cột Tuổi bằng IQR
Q1 = df_clean['Tuổi'].quantile(0.25)
Q3 = df_clean['Tuổi'].quantile(0.75)
IQR = Q3 - Q1
gioi_han_tren = Q3 + 1.5 * IQR
gioi_han_duoi = Q1 - 1.5 * IQR

print(f'Q1={Q1}, Q3={Q3}, IQR={IQR}')
print(f'Giới hạn hợp lệ: [{gioi_han_duoi:.0f}, {gioi_han_tren:.0f}]')

outlier = df_clean[~df_clean['Tuổi'].between(gioi_han_duoi, gioi_han_tren)]
print(f'\nCác dòng có Tuổi ngoại lệ:')
print(outlier[['STT', 'Họ tên', 'Tuổi']])

# Xử lý: thay bằng giá trị trung vị
df_clean.loc[~df_clean['Tuổi'].between(gioi_han_duoi, gioi_han_tren), 'Tuổi'] = trung_vi_tuoi
print(f'\nĐã thay thế giá trị ngoại lệ bằng trung vị ({trung_vi_tuoi})')

In [ ]:
# Phát hiện và xử lý lương <= 0
print('Dòng có lương <= 0:')
print(df_clean[df_clean['Lương'] <= 0][['STT', 'Họ tên', 'Lương']])

df_clean.loc[df_clean['Lương'] <= 0, 'Lương'] = trung_binh_luong
print('Đã thay thế lương <= 0 bằng trung bình')

### 4.4 Xử lý dữ liệu trùng lặp

In [ ]:
# Kiểm tra dòng trùng lặp hoàn toàn
trung_lap = df_clean.duplicated()
print(f'Số dòng trùng lặp hoàn toàn: {trung_lap.sum()}')
print('Các dòng trùng lặp:')
print(df_clean[df_clean.duplicated(keep=False)])

# Xóa trùng lặp, giữ lại dòng đầu tiên
df_clean = df_clean.drop_duplicates(keep='first')
print(f'\nSau khi xóa trùng lặp: {len(df_clean)} dòng')

In [ ]:
# Kiểm tra trùng lặp theo Email (khóa nghiệp vụ)
trung_email = df_clean.duplicated(subset=['Email'], keep=False)
print(f'Trùng lặp theo Email: {trung_email.sum()} dòng')
print(df_clean[trung_email][['STT', 'Họ tên', 'Email']])

### 4.5 Chuẩn hóa định dạng chuỗi

Các phương thức `.str` thường dùng:

| Phương thức | Ý nghĩa |
|------------|----------|
| `.str.strip()` | Xóa khoảng trắng đầu/cuối |
| `.str.lower()` | Chuyển thành chữ thường |
| `.str.upper()` | Chuyển thành chữ hoa |
| `.str.title()` | Viết hoa chữ cái đầu mỗi từ |
| `.str.replace(a, b)` | Thay chuỗi a bằng chuỗi b |
| `.str.contains(pattern)` | Kiểm tra có chứa chuỗi không |

In [ ]:
# Chuẩn hóa họ tên: xóa khoảng trắng thừa, viết hoa chữ cái đầu
print('Trước khi chuẩn hóa:')
print(df_clean['Họ tên'].tolist())

df_clean['Họ tên'] = df_clean['Họ tên'].str.strip().str.title()

print('\nSau khi chuẩn hóa:')
print(df_clean['Họ tên'].tolist())

In [ ]:
# Chuẩn hóa email: chuyển thành chữ thường, xóa khoảng trắng
df_clean['Email'] = df_clean['Email'].str.strip().str.lower()
df_clean['Thành phố'] = df_clean['Thành phố'].str.strip().str.title()

# Kiểm tra định dạng email hợp lệ bằng regex
email_hop_le = df_clean['Email'].str.contains(r'^[\w.-]+@[\w.-]+\.\w+$', na=False)
print(f'Email hợp lệ: {email_hop_le.sum()}/{len(df_clean)}')
print('Email không hợp lệ:')
print(df_clean[~email_hop_le][['STT', 'Họ tên', 'Email']])

In [ ]:
# Kết quả sau làm sạch
print('=== DỮ LIỆU SAU LÀM SẠCH ===')
df_clean

### Bài tập 4.1

Áp dụng các kỹ thuật làm sạch cho DataFrame `df_nv` (bảng nhân viên):

1. Kiểm tra và in ra báo cáo: số dòng null theo từng cột, số dòng trùng lặp
2. Chuẩn hóa cột `Họ đệm` và `Tên`: xóa khoảng trắng thừa, viết hoa đúng chuẩn
3. Kiểm tra có nhân viên nào có `Lương cơ bản` <= 0 không?
4. Tạo cột `Tuổi` từ cột `Năm sinh` (giả sử năm hiện tại là 2025)
5. Lọc ra các nhân viên có `Số ngày phép` > 12 (vi phạm giới hạn) và in danh sách

In [ ]:
# --- KHUNG BÀI TẬP 4.1 ---
df_nv_clean = df_nv.copy()

# 1. Báo cáo null và trùng lặp
print('--- Null ---')
# TODO

print('--- Trùng lặp ---')
# TODO

# 2. Chuẩn hóa Họ đệm, Tên
# TODO

# 3. Kiểm tra lương <= 0
# TODO

# 4. Tạo cột Tuổi
# df_nv_clean['Tuổi'] = ...  # TODO

# 5. Nhân viên có ngày phép > 12
print('Nhân viên có ngày phép > 12:')
# TODO

---
## PHẦN 5: Tạo thêm Cột và Biến đổi Dữ liệu

### 5.1 Tạo cột mới bằng phép tính toán

In [ ]:
df_nv2 = df_nv.copy()

# Tuổi nhân viên
df_nv2['Tuổi'] = 2025 - df_nv2['Năm sinh']

# Tổng thu nhập (lương + phụ cấp)
df_nv2['Tổng thu nhập'] = df_nv2['Lương cơ bản'] + df_nv2['Phụ cấp']

# Tỷ lệ thuế trên lương cơ bản
df_nv2['Tỷ lệ thuế (%)'] = (df_nv2['Thuế TNCN'] / df_nv2['Lương cơ bản'] * 100).round(2)

# Xếp loại lương bằng hàm tự định nghĩa
def xep_loai_luong(luong):
    if luong < 10_000_000:   return 'Thấp'
    elif luong < 20_000_000: return 'Trung bình'
    elif luong < 30_000_000: return 'Khá'
    else:                    return 'Cao'

df_nv2['Mức lương'] = df_nv2['Lương cơ bản'].apply(xep_loai_luong)

# Nhóm tuổi bằng pd.cut — chia khoảng liên tục thành nhãn rời rạc
df_nv2['Nhóm tuổi'] = pd.cut(
    df_nv2['Tuổi'],
    bins=[0, 30, 40, 50, 100],
    labels=['Dưới 30', '30–40', '40–50', 'Trên 50']
)

cols_xem = ['Mã NV', 'Họ đệm', 'Tên', 'Tuổi', 'Nhóm tuổi',
            'Lương cơ bản', 'Mức lương', 'Tổng thu nhập', 'Tỷ lệ thuế (%)']
df_nv2[cols_xem].head(10)

### 5.2 np.where và np.select — Tạo cột có điều kiện

- `np.where(dk, gt_đúng, gt_sai)` — tương đương `if-else` áp dụng toàn bộ cột
- `np.select(danh_sách_dk, danh_sách_gt, default)` — tương đương `if-elif-elif-else`

In [ ]:
# np.where: nhân viên xuất sắc hoặc lương thấp dưới 12 triệu thì có ưu đãi
df_nv2['Có ưu đãi'] = np.where(
    (df_nv2['Đánh giá'] == 'Xuất sắc') | (df_nv2['Lương cơ bản'] < 12_000_000),
    'Có',
    'Không'
)

# np.select: hệ số tăng lương theo đánh giá
conditions = [
    df_nv2['Đánh giá'] == 'Xuất sắc',
    df_nv2['Đánh giá'] == 'Tốt',
    df_nv2['Đánh giá'] == 'Khá',
    df_nv2['Đánh giá'] == 'Trung bình',
]
choices = [1.15, 1.10, 1.05, 1.02]

df_nv2['Hệ số tăng'] = np.select(conditions, choices, default=1.0)
df_nv2['Lương mới']  = (df_nv2['Lương cơ bản'] * df_nv2['Hệ số tăng']).round(-4)

df_nv2[['Mã NV', 'Đánh giá', 'Lương cơ bản', 'Hệ số tăng', 'Lương mới']].head(10)

### 5.3 .apply() — Áp dụng hàm phức tạp lên từng dòng

`apply(func, axis=1)` gọi hàm `func` cho mỗi dòng, nhận vào toàn bộ dòng đó dưới dạng Series.

In [ ]:
def tinh_thuong(row):
    """
    Tính thưởng cuối năm theo đánh giá và số ngày phép đã dùng.
    Quy tắc: Xuất sắc=3 tháng lương, Tốt=2, Khá=1.5, TB=1, Yếu=0.
    Giảm 5% cho mỗi ngày phép đã sử dụng (giả sử tổng 12 ngày/năm).
    """
    he_so_map = {'Xuất sắc': 3.0, 'Tốt': 2.0, 'Khá': 1.5, 'Trung bình': 1.0, 'Yếu': 0.0}
    he_so = he_so_map.get(row['Đánh giá'], 0)
    ngay_da_dung = 12 - row['Số ngày phép']
    he_so_thuc = he_so * max(0, 1 - ngay_da_dung * 0.05)
    return round(row['Lương cơ bản'] * he_so_thuc, -4)

df_nv2['Thưởng cuối năm'] = df_nv2.apply(tinh_thuong, axis=1)

df_nv2[['Mã NV', 'Đánh giá', 'Số ngày phép', 'Lương cơ bản', 'Thưởng cuối năm']].head(10)

### Bài tập 5.1

Thêm vào DataFrame `df_nv2`:

1. Cột `Năm kinh nghiệm`: tính từ ngày vào làm đến 2025
   - **Gợi ý**: `pd.to_datetime(df_nv2['Ngày vào làm'], format='%d/%m/%Y')`
2. Cột `Cấp bậc`: < 2 năm → 'Junior', 2–5 năm → 'Middle', 5–10 năm → 'Senior', > 10 năm → 'Principal'
3. Cột `Phụ cấp đi lại`: nhân viên ngoài Hà Nội và Hồ Chí Minh được cộng thêm 500.000 VND
4. Cột `Lương tổng hợp`: Lương cơ bản + Phụ cấp + Phụ cấp đi lại − Thuế TNCN

In [ ]:
# --- KHUNG BÀI TẬP 5.1 ---

# 1. Năm kinh nghiệm
# TODO

# 2. Cấp bậc
# TODO

# 3. Phụ cấp đi lại
# TODO

# 4. Lương tổng hợp
# TODO

# Kiểm tra kết quả
# df_nv2[['Mã NV', 'Tỉnh thành', 'Năm kinh nghiệm', 'Cấp bậc',
#          'Phụ cấp đi lại', 'Lương tổng hợp']].head(10)

---
## PHẦN 6: Sắp xếp và Lọc Nâng cao

In [ ]:
# Sắp xếp tăng dần — 5 nhân viên lương thấp nhất
df_tang = df_nv.sort_values('Lương cơ bản')
print('5 nhân viên lương thấp nhất:')
print(df_tang[['Mã NV', 'Họ đệm', 'Tên', 'Phòng ban', 'Lương cơ bản']].head())

In [ ]:
# Sắp xếp giảm dần — 5 nhân viên lương cao nhất
df_giam = df_nv.sort_values('Lương cơ bản', ascending=False)
print('5 nhân viên lương cao nhất:')
print(df_giam[['Mã NV', 'Họ đệm', 'Tên', 'Phòng ban', 'Lương cơ bản']].head())

In [ ]:
# Sắp xếp nhiều cột: phòng ban tăng dần, lương giảm dần
df_multi = df_nv.sort_values(
    by=['Phòng ban', 'Lương cơ bản'],
    ascending=[True, False]
)
df_multi[['Mã NV', 'Họ đệm', 'Tên', 'Phòng ban', 'Lương cơ bản']].head(15)

In [ ]:
# Nhân viên lương cao nhất mỗi phòng ban
df_top1 = (
    df_nv
    .sort_values('Lương cơ bản', ascending=False)
    .groupby('Phòng ban')
    .first()
    .reset_index()
)
print('Nhân viên lương cao nhất mỗi phòng ban:')
df_top1[['Phòng ban', 'Mã NV', 'Họ đệm', 'Tên', 'Lương cơ bản']]

In [ ]:
# Xếp hạng lương trong toàn công ty và trong phòng ban
df_nv2['Hạng lương'] = df_nv2['Lương cơ bản'].rank(ascending=False, method='dense').astype(int)
df_nv2['Hạng trong phòng'] = (
    df_nv2.groupby('Phòng ban')['Lương cơ bản']
    .rank(ascending=False, method='dense')
    .astype(int)
)
df_nv2[['Mã NV', 'Họ đệm', 'Tên', 'Phòng ban', 'Lương cơ bản',
        'Hạng lương', 'Hạng trong phòng']].head(12)

---
## PHẦN 7: Thống kê và Phân tích Theo Nhóm — groupby

### 7.1 groupby cơ bản

`groupby` hoạt động theo 3 bước: **Split** (phân nhóm) → **Apply** (tính toán) → **Combine** (tổng hợp kết quả)

Các hàm tổng hợp phổ biến: `count`, `sum`, `mean`, `median`, `min`, `max`, `std`, `first`, `last`

In [ ]:
# Thống kê lương theo phòng ban
thong_ke_pb = df_nv.groupby('Phòng ban')['Lương cơ bản'].agg(
    so_nguoi='count',
    luong_tb='mean',
    luong_min='min',
    luong_max='max',
    luong_tong='sum',
    do_lech_chuan='std'
).round(0)

thong_ke_pb.sort_values('luong_tb', ascending=False)

In [ ]:
# Thống kê nhiều cột cùng lúc bằng .agg() với dict
thong_ke_da_chieu = df_nv.groupby('Phòng ban').agg(
    so_NV=('Mã NV', 'count'),
    luong_tb=('Lương cơ bản', 'mean'),
    phu_cap_tb=('Phụ cấp', 'mean'),
    luong_thuc_tb=('Lương thực lĩnh', 'mean'),
    ngay_phep_tb=('Số ngày phép', 'mean')
).round(0)

thong_ke_da_chieu

In [ ]:
# Phân tích theo 2 chiều: phòng ban và giới tính
phan_tich_2c = df_nv.groupby(['Phòng ban', 'Giới tính']).agg(
    so_nguoi=('Mã NV', 'count'),
    luong_tb=('Lương cơ bản', 'mean')
).round(0)

phan_tich_2c

In [ ]:
# Phân tích đơn hàng theo khu vực và trạng thái
phan_tich_dh = df_dh.groupby(['Khu vực', 'Trạng thái']).agg(
    so_don=('Mã đơn', 'count'),
    tong_doanh_thu=('Thành tiền', 'sum'),
    tb_don=('Thành tiền', 'mean')
).round(0)

phan_tich_dh

### 7.2 transform — Giữ nguyên kích thước DataFrame

Khác với `agg` (giảm số dòng), `transform` trả về Series **cùng kích thước** ban đầu.  
Dùng để tạo cột mới có giá trị tính theo nhóm, ví dụ: điền trung bình nhóm vào từng dòng.

In [ ]:
# Lương trung bình của phòng ban — điền vào từng nhân viên
df_nv2['Lương TB phòng'] = df_nv2.groupby('Phòng ban')['Lương cơ bản'].transform('mean').round(0)

# So sánh lương cá nhân với trung bình phòng ban
df_nv2['Chênh lệch lương'] = df_nv2['Lương cơ bản'] - df_nv2['Lương TB phòng']
df_nv2['Chênh lệch (%)']   = (df_nv2['Chênh lệch lương'] / df_nv2['Lương TB phòng'] * 100).round(2)

cols = ['Mã NV', 'Họ đệm', 'Tên', 'Phòng ban', 'Lương cơ bản',
        'Lương TB phòng', 'Chênh lệch lương', 'Chênh lệch (%)']
df_nv2[cols].sort_values('Chênh lệch (%)', ascending=False).head(10)

### Bài tập 7.1

Thực hiện phân tích bằng `groupby` trên DataFrame `df_dh` (đơn hàng):

1. Thống kê theo **Danh mục sản phẩm**: số đơn, tổng doanh thu, đơn hàng cao nhất
2. Doanh thu **mỗi tháng** (trích xuất tháng từ `Ngày đặt`)
3. **Top 5 nhân viên** có tổng doanh số cao nhất
4. Tỷ lệ **đơn hàng bị hủy** theo từng khu vực
5. Sản phẩm có **số lượng bán trung bình** cao nhất

In [ ]:
# --- KHUNG BÀI TẬP 7.1 ---

# Chuyển Ngày đặt về datetime trước
df_dh['Ngày đặt (dt)'] = pd.to_datetime(df_dh['Ngày đặt'], format='%d/%m/%Y')
df_dh['Tháng'] = df_dh['Ngày đặt (dt)'].dt.month
df_dh['Năm']   = df_dh['Ngày đặt (dt)'].dt.year

# 1. Thống kê theo danh mục
bai1 = ...  # TODO

# 2. Doanh thu mỗi tháng
bai2 = ...  # TODO

# 3. Top 5 nhân viên doanh số
bai3 = ...  # TODO

# 4. Tỷ lệ đơn bị hủy
bai4 = ...  # TODO

# 5. Sản phẩm số lượng TB cao nhất
bai5 = ...  # TODO

---
## PHẦN 8: Kết nối Dữ liệu — merge, join, concat

### 8.1 merge — tương tự SQL JOIN

| Kiểu join | Ý nghĩa |
|-----------|---------|
| `inner` | Chỉ giữ hàng có khóa tồn tại ở cả 2 bảng |
| `left`  | Giữ tất cả hàng bảng trái, NaN nếu không khớp |
| `right` | Giữ tất cả hàng bảng phải |
| `outer` | Giữ tất cả hàng cả 2 bảng |

In [ ]:
# Kết nối đơn hàng với thông tin nhân viên
df_dh_nv = pd.merge(
    df_dh,
    df_nv[['Mã NV', 'Họ đệm', 'Tên', 'Phòng ban', 'Tỉnh thành']],
    on='Mã NV',
    how='left'
)

print(f'df_dh ban đầu  : {df_dh.shape}')
print(f'Sau khi merge  : {df_dh_nv.shape}')
print(f'Đơn chưa tìm thấy NV: {df_dh_nv["Họ đệm"].isnull().sum()}')

df_dh_nv[['Mã đơn', 'Ngày đặt', 'Mã NV', 'Họ đệm', 'Tên',
          'Phòng ban', 'Tên SP', 'Thành tiền']].head(8)

In [ ]:
# Nhân viên bán chạy nhất
doanh_so_nv = (
    df_dh_nv
    .groupby(['Mã NV', 'Họ đệm', 'Tên', 'Phòng ban'])
    .agg(
        so_don=('Mã đơn', 'count'),
        tong_doanh_thu=('Thành tiền', 'sum')
    )
    .reset_index()
    .sort_values('tong_doanh_thu', ascending=False)
)
print('Top 10 nhân viên doanh số cao nhất:')
doanh_so_nv.head(10)

In [ ]:
# concat: nối chồng (theo hàng) hoặc nối ngang (theo cột)
df_nua_1 = df_nv.iloc[:50]
df_nua_2 = df_nv.iloc[50:]

print(f'Phần 1: {df_nua_1.shape}, Phần 2: {df_nua_2.shape}')

# Nối theo hàng (axis=0)
df_noi = pd.concat([df_nua_1, df_nua_2], axis=0, ignore_index=True)
print(f'Sau concat: {df_noi.shape}')

# Nối theo cột (axis=1) — cần cùng số dòng
df_a = df_nv[['Mã NV', 'Họ đệm', 'Tên']].head(5)
df_b = df_nv[['Phòng ban', 'Lương cơ bản']].head(5)
print('\nNối ngang:')
print(pd.concat([df_a, df_b], axis=1))

---
## PHẦN 9: Pivot Table và Phân tích Chéo

### 9.1 pivot_table

Pivot table tóm tắt dữ liệu theo 2 chiều, tương tự PivotTable trong Excel.

```python
pd.pivot_table(df, values, index, columns, aggfunc, fill_value, margins)
```

In [ ]:
# Lương trung bình theo phòng ban và trình độ học vấn
pivot_luong = pd.pivot_table(
    df_nv,
    values='Lương cơ bản',
    index='Phòng ban',
    columns='Trình độ',
    aggfunc='mean',
    fill_value=0
).round(0)

print('Lương trung bình (dòng = Phòng ban, cột = Trình độ):')
pivot_luong

In [ ]:
# Số nhân viên theo phòng ban và đánh giá — thêm cột tổng
pivot_danh_gia = pd.pivot_table(
    df_nv,
    values='Mã NV',
    index='Phòng ban',
    columns='Đánh giá',
    aggfunc='count',
    fill_value=0
)
pivot_danh_gia['TỔNG'] = pivot_danh_gia.sum(axis=1)
print('Số nhân viên theo phòng ban và đánh giá:')
pivot_danh_gia

In [ ]:
# Trực quan hóa pivot bằng heatmap
fig, ax = plt.subplots(figsize=(11, 5))
pivot_num = pivot_luong.replace(0, float('nan'))
im = ax.imshow(pivot_num.values, cmap='YlOrRd', aspect='auto')
plt.colorbar(im, ax=ax, label='Lương cơ bản (VND)')

ax.set_xticks(range(len(pivot_num.columns)))
ax.set_xticklabels(pivot_num.columns, rotation=30, ha='right')
ax.set_yticks(range(len(pivot_num.index)))
ax.set_yticklabels(pivot_num.index)
ax.set_title('Lương trung bình theo Phòng ban và Trình độ')

for i in range(len(pivot_num.index)):
    for j in range(len(pivot_num.columns)):
        val = pivot_num.values[i, j]
        if not np.isnan(val):
            ax.text(j, i, f'{val/1e6:.1f}M', ha='center', va='center', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Tổng doanh thu theo khu vực và danh mục — thêm dòng/cột tổng bằng margins
pivot_doanh_thu = pd.pivot_table(
    df_dh,
    values='Thành tiền',
    index='Khu vực',
    columns='Danh mục',
    aggfunc='sum',
    fill_value=0,
    margins=True,
    margins_name='TỔNG'
).round(0)

print('Tổng doanh thu theo Khu vực và Danh mục:')
pivot_doanh_thu

### Bài tập 9.1

Xây dựng các pivot table sau:

1. **Số lượng đơn hàng** theo từng tháng (cột) và từng năm (hàng), có cột tổng
2. **Tỷ lệ hoàn thành** (Trạng thái == 'Đã giao') theo Khu vực và Tên SP
3. **Lương trung bình** theo Phòng ban (hàng) và Nhóm tuổi (cột)
4. Trực quan hóa pivot số 3 bằng biểu đồ thanh so sánh

In [ ]:
# --- KHUNG BÀI TẬP 9.1 ---

# 1. Số đơn theo tháng và năm
# TODO

# 2. Tỷ lệ hoàn thành
# TODO

# 3. Lương theo phòng ban và nhóm tuổi (cần tạo cột Nhóm tuổi từ df_nv2)
# TODO

# 4. Biểu đồ thanh
# TODO

---
## PHẦN 10: Trực quan hóa Dữ liệu

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Phân tích Lương Nhân viên', fontsize=15, fontweight='bold')

# 1. Histogram phân phối lương
ax1 = axes[0, 0]
ax1.hist(df_nv['Lương cơ bản'] / 1e6, bins=15, color='steelblue', edgecolor='white')
ax1.set_title('Phân phối Lương cơ bản')
ax1.set_xlabel('Lương (triệu VND)')
ax1.set_ylabel('Số nhân viên')
ax1.axvline(df_nv['Lương cơ bản'].mean()/1e6, color='red', linestyle='--', label='Trung bình')
ax1.legend()

# 2. Biểu đồ cột: số nhân viên mỗi phòng ban
ax2 = axes[0, 1]
so_nv_pb = df_nv['Phòng ban'].value_counts()
ax2.bar(range(len(so_nv_pb)), so_nv_pb.values, color='teal')
ax2.set_xticks(range(len(so_nv_pb)))
ax2.set_xticklabels(so_nv_pb.index, rotation=30, ha='right')
ax2.set_title('Số nhân viên theo Phòng ban')
ax2.set_ylabel('Số người')

# 3. Boxplot lương theo phòng ban
ax3 = axes[1, 0]
pb_list = df_nv['Phòng ban'].unique()
data_box = [df_nv[df_nv['Phòng ban']==pb]['Lương cơ bản'].values/1e6 for pb in pb_list]
bp = ax3.boxplot(data_box, labels=pb_list, patch_artist=True)
for patch in bp['boxes']:
    patch.set_facecolor('lightcyan')
ax3.set_xticklabels(pb_list, rotation=30, ha='right')
ax3.set_title('Phân phối Lương theo Phòng ban')
ax3.set_ylabel('Lương (triệu VND)')

# 4. Biểu đồ tròn: tỉ lệ đánh giá
ax4 = axes[1, 1]
dg_ct = df_nv['Đánh giá'].value_counts()
ax4.pie(dg_ct.values, labels=dg_ct.index,
        autopct='%1.1f%%', startangle=90,
        colors=['#2ecc71','#3498db','#f39c12','#e74c3c','#9b59b6'])
ax4.set_title('Tỉ lệ Đánh giá Nhân viên')

plt.tight_layout()
plt.show()

In [ ]:
# Biểu đồ xu hướng doanh thu theo tháng
df_dh['Ngày đặt (dt)'] = pd.to_datetime(df_dh['Ngày đặt'], format='%d/%m/%Y', errors='coerce')
df_dh['Năm-Tháng'] = df_dh['Ngày đặt (dt)'].dt.to_period('M').astype(str)

doanh_thu_thang = (
    df_dh[df_dh['Trạng thái'] == 'Đã giao']
    .groupby('Năm-Tháng')['Thành tiền']
    .sum()
    .reset_index()
    .sort_values('Năm-Tháng')
)

fig, ax = plt.subplots(figsize=(14, 5))
x = range(len(doanh_thu_thang))
ax.plot(x, doanh_thu_thang['Thành tiền']/1e6, marker='o', linewidth=2, color='steelblue')
ax.fill_between(x, doanh_thu_thang['Thành tiền']/1e6, alpha=0.3, color='steelblue')
ax.set_xticks(x)
ax.set_xticklabels(doanh_thu_thang['Năm-Tháng'], rotation=45, ha='right')
ax.set_title('Doanh thu Hàng tháng (Đơn đã giao)')
ax.set_ylabel('Doanh thu (triệu VND)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## PHẦN 11: Xuất Dữ liệu ra File Excel

In [ ]:
# Xuất đơn giản — một sheet
df_nv2.to_excel('ket_qua_nhan_vien.xlsx', index=False)
print('Đã xuất ra ket_qua_nhan_vien.xlsx')

In [ ]:
# Xuất nhiều sheet vào một file Excel duy nhất bằng ExcelWriter
with pd.ExcelWriter('bao_cao_tong_hop.xlsx', engine='openpyxl') as writer:

    df_nv2.to_excel(writer, sheet_name='Nhân viên', index=False)
    thong_ke_pb.to_excel(writer, sheet_name='TK Phòng ban')
    df_dh.to_excel(writer, sheet_name='Đơn hàng', index=False)
    doanh_thu_thang.to_excel(writer, sheet_name='Doanh thu tháng', index=False)
    doanh_so_nv.head(20).to_excel(writer, sheet_name='Top NV doanh số', index=False)

print('Đã xuất ra bao_cao_tong_hop.xlsx')
print('Các sheet: Nhân viên | TK Phòng ban | Đơn hàng | Doanh thu tháng | Top NV doanh số')

In [ ]:
# Xuất với định dạng đẹp bằng openpyxl
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

df_export = thong_ke_pb.reset_index()
with pd.ExcelWriter('bao_cao_dinh_dang.xlsx', engine='openpyxl') as writer:
    df_export.to_excel(writer, sheet_name='Thong_ke', index=False)

wb = load_workbook('bao_cao_dinh_dang.xlsx')
ws = wb['Thong_ke']

# Định dạng dòng tiêu đề
for cell in ws[1]:
    cell.font      = Font(name='Arial', bold=True, color='FFFFFF', size=11)
    cell.fill      = PatternFill('solid', start_color='1F4E79')
    cell.alignment = Alignment(horizontal='center', vertical='center')

# Định dạng số và căn giữa dữ liệu
for row in ws.iter_rows(min_row=2):
    for cell in row:
        if isinstance(cell.value, (int, float)):
            cell.number_format = '#,##0'
        cell.alignment = Alignment(horizontal='center')

# Tự động chỉnh độ rộng cột
for col in ws.columns:
    max_len = max(len(str(c.value)) if c.value else 0 for c in col)
    ws.column_dimensions[get_column_letter(col[0].column)].width = max_len + 4

ws.freeze_panes = 'A2'
wb.save('bao_cao_dinh_dang.xlsx')
print('Đã xuất bao_cao_dinh_dang.xlsx với định dạng đẹp')

---
## PHẦN 12: Bài Toán Tổng Hợp — Báo cáo Nhân sự Hàng Quý

Yêu cầu: Xây dựng báo cáo nhân sự Q4/2025 bao gồm:
- Tổng hợp theo phòng ban
- Danh sách nhân viên cần tăng lương (đánh giá tốt nhưng lương dưới trung bình phòng ban)
- Phân tích chi phí lương toàn công ty
- Xuất ra file Excel 4 sheet có định dạng

In [ ]:
# ---- BƯỚC 1: Chuẩn bị dữ liệu ----
df_bao_cao = df_nv.copy()
df_bao_cao['Tuổi'] = 2025 - df_bao_cao['Năm sinh']
df_bao_cao['Lương TB phòng'] = (
    df_bao_cao.groupby('Phòng ban')['Lương cơ bản'].transform('mean').round(0)
)
df_bao_cao['Dưới trung bình'] = df_bao_cao['Lương cơ bản'] < df_bao_cao['Lương TB phòng']

print('Chuẩn bị dữ liệu hoàn tất')

In [ ]:
# ---- BƯỚC 2: Tổng hợp phòng ban ----
tong_hop_pb = df_bao_cao.groupby('Phòng ban').agg(
    Số_NV              = ('Mã NV', 'count'),
    NV_Nam             = ('Giới tính', lambda x: (x=='Nam').sum()),
    NV_Nữ              = ('Giới tính', lambda x: (x=='Nữ').sum()),
    Lương_TB           = ('Lương cơ bản', 'mean'),
    Lương_Min          = ('Lương cơ bản', 'min'),
    Lương_Max          = ('Lương cơ bản', 'max'),
    Tổng_Chi_Phí_Lương = ('Lương thực lĩnh', 'sum'),
    NV_Xuất_Sắc        = ('Đánh giá', lambda x: (x=='Xuất sắc').sum()),
    NV_Yếu             = ('Đánh giá', lambda x: (x=='Yếu').sum()),
    Tuổi_TB            = ('Tuổi', 'mean')
).round(0)

print('=== TỔNG HỢP THEO PHÒNG BAN ===')
tong_hop_pb

In [ ]:
# ---- BƯỚC 3: Danh sách nhân viên cần xem xét tăng lương ----
can_tang_luong = df_bao_cao[
    df_bao_cao['Đánh giá'].isin(['Xuất sắc', 'Tốt']) &
    df_bao_cao['Dưới trung bình']
][['Mã NV', 'Họ đệm', 'Tên', 'Phòng ban', 'Chức vụ', 'Đánh giá',
   'Lương cơ bản', 'Lương TB phòng']].copy()

can_tang_luong['Mức đề xuất tăng'] = (
    (can_tang_luong['Lương TB phòng'] - can_tang_luong['Lương cơ bản']) * 0.8
).round(-4)

can_tang_luong = can_tang_luong.sort_values('Mức đề xuất tăng', ascending=False)

print(f'Số nhân viên đề xuất tăng lương: {len(can_tang_luong)}')
can_tang_luong

In [ ]:
# ---- BƯỚC 4: Phân tích chi phí lương ----
tong_chi_phi = df_bao_cao['Lương thực lĩnh'].sum()
chi_phi_theo_pb = (
    df_bao_cao.groupby('Phòng ban')['Lương thực lĩnh']
    .sum()
    .sort_values(ascending=False)
)

print(f'Tổng chi phí lương/tháng: {tong_chi_phi:,.0f} VND')
print(f'Tổng chi phí lương/năm  : {tong_chi_phi*12:,.0f} VND')
print()
for pb, cp in chi_phi_theo_pb.items():
    pct = cp / tong_chi_phi * 100
    print(f'  {pb:<15}: {cp:>15,.0f} VND  ({pct:.1f}%)')

In [ ]:
# ---- BƯỚC 5: Xuất báo cáo ra Excel ----
with pd.ExcelWriter('bao_cao_nhan_su_Q4_2025.xlsx', engine='openpyxl') as writer:

    tong_hop_pb.reset_index().to_excel(
        writer, sheet_name='1_Tổng hợp Phòng ban', index=False)

    can_tang_luong.to_excel(
        writer, sheet_name='2_Đề xuất Tăng lương', index=False)

    df_bao_cao.sort_values(['Phòng ban','Lương cơ bản'],
                           ascending=[True,False]).to_excel(
        writer, sheet_name='3_Chi tiết Nhân viên', index=False)

    chi_phi_theo_pb.reset_index().rename(
        columns={'Lương thực lĩnh':'Chi phí tháng'}
    ).to_excel(writer, sheet_name='4_Chi phí Lương', index=False)

print('Đã xuất bao_cao_nhan_su_Q4_2025.xlsx')
print('4 sheet: Tổng hợp Phòng ban | Đề xuất Tăng lương | Chi tiết Nhân viên | Chi phí Lương')

---
## TỔNG KẾT BÀI THỰC HÀNH

| Phần | Nội dung chính | Điểm |
|------|---------------|------|
| 3 | Chọn lọc, lọc dữ liệu | 1.5 |
| 4 | Làm sạch dữ liệu | 2.0 |
| 5 | Tạo cột, biến đổi dữ liệu | 1.5 |
| 7 | Phân tích theo nhóm | 2.0 |
| 9 | Pivot table | 1.5 |
| 12 | Bài toán tổng hợp | 1.5 |
| **Tổng** | | **10** |

---

## Tài liệu tham khảo

- Trang chủ chính thức: https://pandas.pydata.org/docs/
- Cheat sheet: https://pandas.pydata.org/Pandas_Cheat_Sheet.pdf
- API Reference: https://pandas.pydata.org/docs/reference/index.html
- User Guide: https://pandas.pydata.org/docs/user_guide/index.html

---

> Lưu ý: Trước khi nộp bài, chạy **Kernel → Restart & Run All** để đảm bảo toàn bộ code chạy không lỗi.